In [19]:
from ultralytics import YOLO
import cv2
import random

In [20]:
def generate_random_color():
        return tuple(random.randint(0, 255) for _ in range(3))

In [21]:
model = YOLO('yolov8m-face.pt')
video_path = 'IMG_2039.MP4'

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, fps, (width, height))

color_map_for_track_id = {}

while True:
    ret, frame = cap.read()
    
    if not ret:
        break

    bbox_map = [369, 17, 1073, 726]
    frame = frame[bbox_map[1]:bbox_map[1]+bbox_map[3], bbox_map[0]:bbox_map[0]+bbox_map[2]]

    results = model.track(frame, tracker='bytetrack.yaml', persist=True, verbose=False)

    try:
        dets = results[0].boxes.data.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().tolist()
    except:
        cv2.imshow('frame', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        out.write(frame)
        continue

    for det, track_id in zip(dets, track_ids):
        track_id = int(track_id)
        x1, y1, x2, y2, *res = map(int, det)

        color = None
        if track_id not in color_map_for_track_id:
            color_map_for_track_id[track_id] = generate_random_color()
            color = color_map_for_track_id[track_id]
        else:
            color = color_map_for_track_id[track_id]

        cv2.rectangle(frame, (x1, y1), (x2, y2), color_map_for_track_id[track_id], 2)
        cv2.putText(frame, f'{track_id}', (x1, y1), cv2.FONT_HERSHEY_SIMPLEX, 1, color_map_for_track_id[track_id], 2)

    out.write(frame)
    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()